<a href="https://colab.research.google.com/github/ravitkurakula/RaviGPT/blob/main/RaviGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

In [4]:
with open("/content/drive/MyDrive/RaviGPT/data/tiny-shakespeare.txt", 'r', encoding = "utf-8") as f :
  text = f.read()

  print(len(text))
  print(text[:500])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
class CharTokenizer :
  def __init__(self,text) :
    self.char = sorted(list(set(text)))
    self.stoi = {ch : i for i,ch in enumerate(self.char)}
    self.itos = {i : ch for i,ch in enumerate(self.char)}

  @property
  def vocab_size(self):
    return len(self.char)

  def encode(self,text) :
    return [self.stoi[ch] for ch in text]

  def decode(self,tokens):
    return ''.join(self.itos[token] for token in tokens)

In [6]:
tokenizer = CharTokenizer(text)

In [7]:
class TextDataset :
  def __init__(self,data,block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self,idx):
    x = self.data[idx : idx+self.block_size]
    y = self.data[idx+1 : idx+self.block_size+1]

    return x,y

In [8]:
data = tokenizer.encode(text)
dataset = TextDataset(data,block_size=64)


In [9]:
def get_batch(dataset,batch_size):
  indices= torch.randint(len(dataset), (batch_size,))
  x= [dataset[i][0] for i in indices]
  y= [dataset[i][1] for i in indices]

  x= torch.tensor(x,dtype = torch.long)
  y = torch.tensor(y,dtype = torch.long)

  return x,y


In [11]:
x,y = get_batch(dataset,batch_size=32)

print(" x shape : ", x.shape)
print("y  shape : ", y.shape)
print(tokenizer.decode(x[0].tolist()))
print(tokenizer.decode(y[0].tolist()))

 x shape :  torch.Size([32, 64])
y  shape :  torch.Size([32, 64])
; I have the most cause
to be glad of yours.

Roman:
Well, let u
 I have the most cause
to be glad of yours.

Roman:
Well, let us


In [12]:
block_size = 64
batch_size = 32
d_model = 128
n_heads = 4
n_layers = 4
dropout = 0.1

vocab_size  = tokenizer.vocab_size